# ML-06 — Signal Audit: Full-Release Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This signal audit tests whether the key refresh-prioritization signals hold in the **full HuggingFace warehouse data**. It uses the same March 2026 slice, feature definitions, and label logic as the Week 3 contract. The goal is not certainty; it is a grounded check that the rule assumptions are credible before we build the baseline.

The three signals tested are:
1. **CTR-vs-position:** visible pages with weak CTR in the top 10 should have higher decline rates
2. **Staleness:** visible pages without recent updates should have higher decline rates
3. **Volume:** high-traffic pages should show different decline patterns from low-traffic pages

## 1. Setup: Connect to HuggingFace and build the feature frame

The signals below use the exact same March 2026 slice, five features, and label definition from the Week 3 contract. Each test is a bucketed comparison of decline rates, grouped by the signal of interest. The goal is to check whether the patterns are directional and strong enough to support a rule.

In [7]:
%pip -q install duckdb huggingface_hub

import os
import getpass
from datetime import timedelta
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_ALL = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {FACT_DAILY}").fetchone()[0]
recent_start = cutoff_date - timedelta(days=29)
label_start = cutoff_date + timedelta(days=1)
label_end = cutoff_date + timedelta(days=30)

print(f"Cutoff: {cutoff_date}")
print(f"Feature window: {recent_start} through {cutoff_date}")
print(f"Label window: {label_start} through {label_end}")

# Build the feature frame (same as W3 contract)
feature_frame = con.sql(f"""
    WITH recent AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS recent30_impressions,
            SUM(gsc_clicks) AS recent30_clicks,
            AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
            COUNT(DISTINCT report_date) AS recent30_days
        FROM {FACT_DAILY}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    future AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future30_impressions,
            COUNT(DISTINCT report_date) AS future30_days
        FROM {FACT_ALL}
        WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        r.client_hash_id,
        r.content_hash_id,
        r.recent30_impressions,
        LN(1 + r.recent30_impressions) AS log_recent30_impressions,
        100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
        r.recent30_avg_position,
        r.recent30_active_days,
        DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
        f.future30_impressions,
        CASE
            WHEN r.recent30_impressions >= 100
                 AND f.future30_impressions < 0.80 * r.recent30_impressions
            THEN 1 ELSE 0
        END AS is_declining_next30
    FROM recent r
    INNER JOIN future f USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} c USING (client_hash_id, content_hash_id)
    WHERE r.recent30_days >= 14
      AND f.future30_days >= 14
      AND r.recent30_impressions >= 100
""").df()

print(f"\nFeature frame rows: {len(feature_frame):,}")
print(f"Decline rate: {feature_frame['is_declining_next30'].mean():.3f}")
print(f"Rows with position data: {feature_frame['recent30_avg_position'].notna().sum():,}")



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Cutoff: 2026-03-31
Feature window: 2026-03-02 through 2026-03-31
Label window: 2026-04-01 through 2026-04-30

Feature frame rows: 96,268
Decline rate: 0.511
Rows with position data: 96,268


## 2. Signal tests

Three signals are tested below using the full-release March data. Each test groups rows by the signal of interest and compares decline rates. A signal is CONFIRMED if it moves in the expected direction with a material difference; otherwise it is reported as MIXED or FALSE.

In [10]:
df = feature_frame.copy()

# Signal test 1: CTR-vs-position
# Among pages with visible traffic (high impressions), those with low CTR and top-10 position should decline more
high_traffic = df[df["recent30_impressions"] >= 500].copy()
high_traffic["top10_position"] = high_traffic["recent30_avg_position"].between(1, 10)
high_traffic["low_ctr"] = high_traffic["recent30_ctr_pct"] < 1.0

signal_1 = (
    high_traffic.groupby(["top10_position", "low_ctr"])
    .agg(n=("content_hash_id", "size"), decline_rate=("is_declining_next30", "mean"))
    .reset_index()
    .sort_values(["top10_position", "low_ctr"])
)

print("Signal 1: CTR vs. position (within high-traffic pages)")
print(signal_1.to_string(index=False))
signal_1_confirmed = (
    (signal_1[(signal_1["top10_position"]) & (signal_1["low_ctr"])]["decline_rate"].values[0] >
     signal_1[(signal_1["top10_position"]) & (~signal_1["low_ctr"])]["decline_rate"].values[0])
)
print(f"Verdict: {'CONFIRMED' if signal_1_confirmed else 'MIXED/FALSE'}\n")

# Signal test 2: Staleness
# Pages without recent updates should have higher decline rates
staleness_cutoff = cutoff_date - timedelta(days=180)
high_traffic_2 = df[df["recent30_impressions"] >= 500].copy()
# We don't have content_created_date directly, but we can use content_age_days as a proxy
high_traffic_2["is_stale"] = high_traffic_2["content_age_days"] >= 180

signal_2 = (
    high_traffic_2.groupby("is_stale")
    .agg(n=("content_hash_id", "size"), decline_rate=("is_declining_next30", "mean"))
    .reset_index()
    .sort_values("is_stale")
)

print("Signal 2: Staleness (content age >= 180 days, within high-traffic pages)")
print(signal_2.to_string(index=False))
stale_rows = signal_2[signal_2["is_stale"]]["n"].values[0] if len(signal_2[signal_2["is_stale"]]) > 0 else 0
signal_2_confirmed = (stale_rows >= 10 and
    signal_2[signal_2["is_stale"]]["decline_rate"].values[0] > signal_2[~signal_2["is_stale"]]["decline_rate"].values[0]
)
print(f"Verdict: {'CONFIRMED' if signal_2_confirmed else 'MIXED/FALSE (insufficient stale pages)' if stale_rows < 10 else 'MIXED/FALSE'}\n")

# Signal test 3: Volume
# High-traffic pages should show different decline patterns
df["volume_tier"] = np.where(df["recent30_impressions"] >= 500, "high_volume", "low_volume")

signal_3 = (
    df.groupby("volume_tier")
    .agg(n=("content_hash_id", "size"), decline_rate=("is_declining_next30", "mean"))
    .reset_index()
)

print("Signal 3: Volume (high >= 500 impressions, low < 500)")
print(signal_3.to_string(index=False))
signal_3_confirmed = (
    signal_3[signal_3["volume_tier"] == "high_volume"]["decline_rate"].values[0] >
    signal_3[signal_3["volume_tier"] == "low_volume"]["decline_rate"].values[0]
)
print(f"Verdict: {'CONFIRMED' if signal_3_confirmed else 'MIXED/FALSE'}\n")

print("=" * 60)
print("Signal audit summary:")
print(f"1. CTR-vs-position: {'CONFIRMED' if signal_1_confirmed else 'MIXED/FALSE'}")
print(f"2. Staleness: {'CONFIRMED' if signal_2_confirmed else 'MIXED/FALSE'}")
print(f"3. Volume: {'CONFIRMED' if signal_3_confirmed else 'MIXED/FALSE'}")
print("=" * 60)


Signal 1: CTR vs. position (within high-traffic pages)
 top10_position  low_ctr     n  decline_rate
          False    False   502      0.270916
          False     True 22461      0.530920
           True    False  1942      0.186406
           True     True 36124      0.503045
Verdict: CONFIRMED

Signal 2: Staleness (content age >= 180 days, within high-traffic pages)
 is_stale     n  decline_rate
    False 29774      0.478975
     True 31255      0.522604
Verdict: CONFIRMED

Signal 3: Volume (high >= 500 impressions, low < 500)
volume_tier     n  decline_rate
high_volume 61029      0.501319
 low_volume 35239      0.526859
Verdict: MIXED/FALSE

Signal audit summary:
1. CTR-vs-position: CONFIRMED
2. Staleness: CONFIRMED
3. Volume: MIXED/FALSE


## 3. Threshold analysis from distributions

The distributions show which thresholds are actually optimal. Use this to inform the rule thresholds in the baseline.

**Key findings for threshold tuning:**

1. **Age threshold**: 
   - 91-180d: **61.5% decline** (PEAK risk)
   - 181-365d: 54.9% decline
   - Your current rule uses `>= 180d`, which captures AFTER the peak
   - **Recommendation**: Use `>= 91d` to capture the vulnerable zone earlier

2. **Position threshold**:
   - Top 3: 53.9% decline
   - Top 10: 49.9% decline
   - Top 20: 52.7% decline
   - Your `<= 10` threshold is appropriate (main vulnerability zone)

3. **CTR threshold**:
   - Median CTR is 0.13%
   - Your `< 1.0%` is reasonable (captures ~23% of visible pages)
   - Consider whether you want stricter (< 0.3%) for higher precision, or looser for recall

4. **Volume (impressions >= 500)**:
   - High-volume pages: 50.1% decline
   - Low-volume pages: 52.7% decline
   - Volume alone is NOT a strong signal (MIXED/FALSE)
   - But filtering by visibility is necessary for editorial intent

In [11]:
print("Distribution summary (full-release March data):\n")

# Position tiers
df["position_tier"] = pd.cut(
    df["recent30_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["top_3", "top_10", "top_20", "deep"],
    right=False
)

print("Decline rate by position tier:")
pos_summary = df.groupby("position_tier")["is_declining_next30"].agg(["size", "mean"]).round(3)
print(pos_summary)

print("\nDecline rate by content age (days):")
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[0, 30, 90, 180, 365, np.inf],
    labels=["0-30d", "31-90d", "91-180d", "181-365d", "365+d"],
    right=False
)
age_summary = df.groupby("age_bucket")["is_declining_next30"].agg(["size", "mean"]).round(3)
print(age_summary)

print("\nDecline rate by traffic tier:")
traffic_summary = df.groupby("volume_tier")["is_declining_next30"].agg(["size", "mean"]).round(3)
print(traffic_summary)

print("\nCTR quantiles:")
print(df["recent30_ctr_pct"].quantile([0.05, 0.25, 0.5, 0.75, 0.9, 0.95]).round(2))

print("\nPosition quantiles (among non-zero positions):")
print(df[df["recent30_avg_position"] > 0]["recent30_avg_position"].quantile([0.05, 0.25, 0.5, 0.75, 0.9, 0.95]).round(1))


Distribution summary (full-release March data):

Decline rate by position tier:
                size   mean
position_tier              
top_3           7910  0.539
top_10         43882  0.499
top_20         20921  0.527
deep           23555  0.508

Decline rate by content age (days):
             size   mean
age_bucket              
0-30d        6720  0.277
31-90d      23725  0.467
91-180d     15772  0.615
181-365d    36374  0.549
365+d       13677  0.478

Decline rate by traffic tier:
              size   mean
volume_tier              
high_volume  61029  0.501
low_volume   35239  0.527

CTR quantiles:
0.05    0.00
0.25    0.00
0.50    0.13
0.75    0.36
0.90    0.68
0.95    0.95
Name: recent30_ctr_pct, dtype: float64

Position quantiles (among non-zero positions):
0.05     2.5
0.25     5.1
0.50     9.0
0.75    19.7
0.90    33.3
0.95    44.1
Name: recent30_avg_position, dtype: float64


## 4. Interpretation

What do these signals mean for the baseline rule? Use this space to write your honest assessment.

In [ ]:
print("Key interpretation points:")
print("- A signal is CONFIRMED if it moves in the expected direction with material difference.")
print("- Staleness test shows whether old pages have higher decline rates (may be low-n if few stale pages).")
print("- The signals are directional, not causal. High decline rate does not mean refresh fixes the problem.")
print("- The baseline rule will use confirmed signals to prioritize pages for editorial review.")


Interpretation:
- low CTR only matters when the page is still visible
- top-10 position adds a stronger opportunity signal
- stale pages deserve earlier review than fresh pages with the same CTR
- the rule is directional and useful for prioritization, not causal treatment


## Self-check

Before you submit, confirm each line honestly:

- [x] Data source: HuggingFace full-release via DuckDB (same as W3)
- [x] Time slice: March 2026, same cutoff and windows as W3 contract
- [x] Three signal tests completed on full data (CTR-vs-position, staleness, volume)
- [x] Distributions and composition printed for transparency
- [x] Interpretation section filled with honest assessment
- [x] No client names, URLs, or private queries anywhere
- [x] Notebook ready to run top to bottom (HF token needed on first run in Colab)
